In [ ]:
#from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import pickle
import pandas as pd
from utils import tld_bucket


In [7]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kaggleprollc/phishing-url-websites-dataset-phiusiil")

print("Path to dataset files:", path)

ModuleNotFoundError: No module named 'kagglehub'

In [ ]:
#df=pd.read_csv("/kaggle/input/phishing-url-websites-dataset-phiusiil/PhiUSIIL_Phishing_URL_Dataset.csv")

In [4]:
df1 = pd.DataFrame({'url': df.URL,'lebel':df.label})

In [5]:
df1.duplicated().sum()

np.int64(425)

In [6]:
df1.drop_duplicates(inplace=True)

In [7]:
from urllib.parse import urlparse
import ipaddress

In [8]:
def get_tld(domain):
    parts = domain.split('.')
    if len(parts) >= 2:
        # Check for common two-part TLDs like .co.uk, .ac.uk, .gov.uk
        if parts[-2] in ['co', 'ac', 'gov'] and len(parts) >= 3:
            return '.' + parts[-2] + '.' + parts[-1]
        # Handle cases like .com, .org, etc.
        return '.' + parts[-1]
    return '' # Return empty string for invalid domains

def is_ip_address(domain):
    try:
        ipaddress.ip_address(domain)
        return 1
    except ValueError:
        return 0

def extract_features(url: str):
    parsed = urlparse(url)
    domain = parsed.netloc

    url_length = len(url)
    domain_length = len(domain)

    is_ip = is_ip_address(domain)

    tld = get_tld(domain)
    tld_length = len(tld)
    tld_risk = tld_bucket(tld) # Assuming tld_bucket is available from utils

    # Number of subdomains, excluding www and the main domain/tld parts
    # This assumes a structure like sub.domain.tld or www.sub.domain.tld
    subdomains = 0
    if domain:
        domain_parts = domain.split('.')
        # Remove known TLD parts based on get_tld logic for more accurate subdomain count
        if tld:
            # Try to remove the TLD part(s) from the end
            if tld.count('.') == 2 and domain_parts[-2:] == tld.split('.')[1:]:
                remaining_parts = domain_parts[:-2]
            elif tld.count('.') == 1 and domain_parts[-1:] == tld.split('.')[1:]:
                remaining_parts = domain_parts[:-1]
            else:
                remaining_parts = domain_parts # Fallback if TLD logic doesn't match easily
        else:
            remaining_parts = domain_parts

        # Filter out 'www' if present in remaining_parts
        filtered_parts = [p for p in remaining_parts if p != 'www']

        # The number of subdomains is essentially the count of parts before the main domain name
        # if there are at least two parts (main_domain.tld), then anything before that is a subdomain.
        # If the domain is just 'example.com' (2 parts), subdomains=0
        # If 'sub.example.com' (3 parts), subdomains=1
        # If 'www.sub.example.com' (4 parts), subdomains=1 (after removing 'www')
        if len(filtered_parts) > 1: # At least domain and potential subdomains
            subdomains = len(filtered_parts) - 1
        else:
            subdomains = 0

    obfuscated_chars = url.count('%') + url.count('\\x')
    has_obfuscation = 1 if obfuscated_chars > 0 else 0
    obfuscation_ratio = obfuscated_chars / url_length if url_length else 0

    letters = sum(c.isalpha() for c in url)
    digits = sum(c.isdigit() for c in url)

    letter_ratio = letters / url_length if url_length else 0
    digit_ratio = digits / url_length if url_length else 0

    equals = url.count('=')
    qmark = url.count('?')
    ampersand = url.count('&')

    special_chars = sum(c in ['@', '-', '_', '%', '=', '?', '&', '.', '/', ':'] for c in url) # Added common URL special chars
    special_ratio = special_chars / url_length if url_length else 0

    is_https = 1 if parsed.scheme == 'https' else 0

    return [
        url_length,
        domain_length,
        is_ip,
        tld_length,
        subdomains,
        has_obfuscation,
        obfuscated_chars,
        obfuscation_ratio,
        letters,
        letter_ratio,
        digits,
        digit_ratio,
        equals,
        qmark,
        ampersand,
        special_chars,
        special_ratio,
        is_https,
        tld_risk
    ]

In [9]:
# Apply the feature extraction function
features = df1['url'].apply(extract_features)

# Create a DataFrame from the extracted features
feature_names = [
    'url_length',
    'domain_length',
    'is_ip',
    'tld_length',
    'subdomains',
    'has_obfuscation',
    'obfuscated_chars',
    'obfuscation_ratio',
    'letters',
    'letter_ratio',
    'digits',
    'digit_ratio',
    'equals',
    'qmark',
    'ampersand',
    'special_chars',
    'special_ratio',
    'is_https',
    'tld_risk'
]

df_features = pd.DataFrame(features.tolist(), columns=feature_names, index=df1.index)

# Concatenate the new features with the original df1
df1 = pd.concat([df1, df_features], axis=1)

print("Features extracted and added to df1.")
display(df1.head())

Features extracted and added to df1.


,url,lebel,url_length,domain_length,is_ip,tld_length,subdomains,has_obfuscation,obfuscated_chars,obfuscation_ratio,...,letter_ratio,digits,digit_ratio,equals,qmark,ampersand,special_chars,special_ratio,is_https,tld_risk
0,https://www.southbankmosaics.com,1,32,24,0,4,0,0,0,0.0,...,0.843750,0,0.0,0,0,0,5,0.156250,1,4
1,https://www.uni-mainz.de,1,24,16,0,3,0,0,0,0.0,...,0.750000,0,0.0,0,0,0,6,0.250000,1,3
2,https://www.voicefmradio.co.uk,1,30,22,0,6,0,0,0,0.0,...,0.800000,0,0.0,0,0,0,6,0.200000,1,4
3,https://www.sfnmjournal.com,1,27,19,0,4,0,0,0,0.0,...,0.814815,0,0.0,0,0,0,5,0.185185,1,4
4,https://www.rewildingargentina.org,1,34,26,0,4,0,0,0,0.0,...,0.852941,0,0.0,0,0,0,5,0.147059,1,4


In [1]:
df.head()

NameError: name 'df' is not defined

In [10]:
df=df1.copy()

In [11]:
df.duplicated().sum()

np.int64(0)

In [12]:
df.drop(columns=['url'],inplace=True)

In [3]:
df=pd.read_csv("cleaned_data (1).csv")

In [4]:
df.head()

,lebel,url_length,domain_length,is_ip,tld_length,subdomains,has_obfuscation,obfuscated_chars,obfuscation_ratio,letters,letter_ratio,digits,digit_ratio,equals,qmark,ampersand,special_chars,special_ratio,is_https,tld_risk
0,1,32,24,0,4,0,0,0,0.0,27,0.843750,0,0.0,0,0,0,5,0.156250,1,4
1,1,24,16,0,3,0,0,0,0.0,18,0.750000,0,0.0,0,0,0,6,0.250000,1,3
2,1,30,22,0,6,0,0,0,0.0,24,0.800000,0,0.0,0,0,0,6,0.200000,1,4
3,1,27,19,0,4,0,0,0,0.0,22,0.814815,0,0.0,0,0,0,5,0.185185,1,4
4,1,34,26,0,4,0,0,0,0.0,29,0.852941,0,0.0,0,0,0,5,0.147059,1,4


In [5]:
X=df.drop(columns=['lebel'])
y=df['lebel']


In [6]:
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [7]:
print(x_train.shape,x_test.shape,y_train.shape,y_test.shape)

(188296, 19) (47074, 19) (188296,) (47074,)


In [8]:
preprocess=ColumnTransformer([
    ("num",Pipeline([
        ('scaler',StandardScaler())
    ]),x_train.columns)
])

In [9]:
pipe=make_pipeline(preprocess,SVC(kernel='rbf',class_weight='balanced',probability=True))

In [10]:
pipe.fit(x_train,y_train)


,steps,"[('columntransformer', ...), ('svc', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [11]:
prediction=pipe.predict(x_test)
prediction_prob=pipe.predict_proba(x_test)[:, 1]
prediction= (prediction_prob> 0.3).astype(int)
prediction_report=classification_report(y_test,prediction,output_dict=True)
prediction_df=pd.DataFrame(prediction_report).transpose()

In [12]:
prediction_df


,precision,recall,f1-score,support
0,0.990234,0.972433,0.981253,19915.00000
1,0.980049,0.992967,0.986466,27159.00000
accuracy,0.984280,0.984280,0.984280,0.98428
macro avg,0.985141,0.982700,0.983859,47074.00000
weighted avg,0.984358,0.984280,0.984260,47074.00000


In [13]:
print(prediction_prob)

[1.15270942e-05 9.86308225e-01 9.85872563e-01 ... 1.00000000e+00
 1.01072666e-04 9.85842185e-01]


In [14]:
y_train_pred = pipe.predict(x_train)
print("Train Accuracy:", accuracy_score(y_train, y_train_pred))
print("Test Accuracy:", accuracy_score(y_test,prediction))

Train Accuracy: 0.9843172451884268
Test Accuracy: 0.984280069677529


In [16]:
conf=pd.DataFrame(confusion_matrix(y_test,prediction), columns=['Predicted Legit','Predicted Phishing'],index=['True Legit','True Phishing'])


In [18]:
conf

,Predicted Legit,Predicted Phishing
True Legit,19366,549
True Phishing,191,26968


In [17]:
# Save the trained SVM model to a file
filename = 'Phish_guard_model.pkl'
pickle.dump(pipe, open(filename, 'wb'))

print(f"SVM model successfully saved to {filename}")

SVM model successfully saved to Phish_guard_model.pkl
